In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def optimize_aleatoric(model, img):
    sigma_alatoric = nn.Parameter(torch.tensor(0.1), requires_grad=True) * torch.eye() 
    optimizer = torch.optim.Adam([sigma_alatoric], lr=0.01)
    
    for _ in range(epochs):
        optimizer.zero_grad()
        mu, sigma, kl = model(img, sigma_alatoric)
        
        # Option 1: optimize NLL
        loss = mc_nll(mu, sigma, kl) + 0.01 * kl

        # Option 1: optimize aleatoric uncertainty
        mu_samples = mu + sigma * torch.randn_like(mu)
        aleatoric = torch.mean(F.softmax(mu_samples, dim=1), dim=0)
        loss = torch.abs(aleatoric - sigma_alatoric)

        loss.backward()
        optimizer.step()
    